In [ ]:
import pandas as pd

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [ ]:
df = pd.read_csv("IMDB Dataset.csv")

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.shape

(50000, 2)

In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(49582, 2)

In [ ]:
df["review"] = df["review"].str.lower()

In [ ]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [ ]:
import re # Regular Expression

In [ ]:
def remove_urls(text):
  text = re.sub(r"http\S+" , "", text)
  return text



df["review"] = df["review"].apply(remove_urls)

In [ ]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [ ]:
def remove_punctuations(text):
  text = re.sub(r"[^A-Za-z0-9\s]" , "", text)
  return text

df["review"] = df["review"].apply(remove_punctuations)

In [ ]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [ ]:
def remove_html(text):
  text = re.sub(r"<.*?>" , "", text)
  return text

df["review"] = df["review"].apply(remove_html)

In [ ]:
import nltk # Natural Language Toolkit

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text = text.replace(word, "")

  return text

df["review"] = df["review"].apply(remove_stopwords)

In [ ]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [ ]:
from nltk.stem import PorterStemmer

In [ ]:
def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []

  tokens = word_tokenize(text)
  for token in tokens:
    stemmed_token = ps.stem(token)
    stemmed_words.append(stemmed_token)

  return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [ ]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [ ]:
y = df["sentiment"]

In [ ]:
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


In [ ]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [ ]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [ ]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057140 stored elements and shape (49582, 5000)>

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape

(39665, 5000)

In [ ]:
X_test.shape

(9917, 5000)

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
  def __init__(self, input_size, hidden_size=128, num_layers=1):
          super().__init__()

          self.hidden_size = hidden_size
          self.num_layers = num_layers

          self.rnn = nn.RNN(input_size,
                            hidden_size,
                            num_layers,
                            batch_first = True)

          self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
    h0 = torch.zeros(self.num_layers,
                   x.size(0),
                   self.hidden_size)

    out, _ = self.rnn(x, h0)

    out = self.fc(out[:, -1, :])
    return out

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
epochs = 10

for epoch in range(epochs):
  model.train()

  for Xb, yb in train_loader:
    optimizer.zero_grad()

    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)

    outputs = torch.sigmoid(outputs.squeeze())

    loss = criterion(outputs, yb)

    loss.backward()
    optimizer.step()

  print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.3771834969520569
epoch = 2/10 and loss = 0.3190762996673584
epoch = 3/10 and loss = 0.23797054588794708
epoch = 4/10 and loss = 0.3831157684326172
epoch = 5/10 and loss = 0.23898597061634064
epoch = 6/10 and loss = 0.23554007709026337
epoch = 7/10 and loss = 0.41207411885261536
epoch = 8/10 and loss = 0.27192607522010803
epoch = 9/10 and loss = 0.2519383132457733
epoch = 10/10 and loss = 0.2305251508951187


In [ ]:
model.eval()

with torch.no_grad():
  correct_vals = 0
  tot_vals = 0

  for Xb, yb in test_loader:
    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)

    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

    tot_vals += yb.size(0)
    correct_vals += (predicted == yb).sum().item()
print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.76182313199556
